# Java Records - Sıfırdan Advanced Səviyyəyə

Bu notebook Java kernel (IJava) üçündür.
Hər code cell-də yalnız Java kodu var və `Run` etdikdə nəticə aşağıda çıxır.

In [ ]:
record Student(String name, int age) {}
var s = new Student("Aylin", 20);
System.out.println(s);
System.out.println("name=" + s.name() + ", age=" + s.age());

## Validation (Compact Constructor)

In [ ]:
record User(String username, String email, int age) {
    public User {
        if (username == null || username.isBlank()) throw new IllegalArgumentException("username boş ola bilməz");
        if (email == null || !email.contains("@")) throw new IllegalArgumentException("email formatı yanlışdır");
        if (age < 0 || age > 120) throw new IllegalArgumentException("yaş aralıqdan kənardır");
    }
}
System.out.println(new User("murad", "murad@example.com", 21));
try {
    new User("", "bad", -5);
} catch (Exception e) {
    System.out.println("Validation xətası: " + e.getMessage());
}

## Əlavə Konstruktor və Metod

In [ ]:
record Money(String currency, long amountInCents) {
    public Money(String currency) { this(currency, 0L); }
}

record Rectangle(double width, double height) {
    double area() { return width * height; }
}

System.out.println(new Money("AZN"));
System.out.println(new Money("USD", 1599));
System.out.println("Area=" + new Rectangle(5, 3).area());

## Defensive Copy

In [ ]:
import java.util.*;

record Course(String title, List<String> topics) {
    public Course {
        topics = List.copyOf(topics);
    }
}

var src = new ArrayList<String>();
src.add("OOP");
src.add("Collections");

var c = new Course("Java", src);
System.out.println("Mənbə dəyişməzdən əvvəl: " + c.topics());
src.add("Streams");
System.out.println("Mənbə dəyişəndən sonra: " + c.topics());

## equals/hashCode və Map Açarı

In [ ]:
import java.util.*;

record Point(int x, int y) {}
record ProductKey(String sku, String region) {}

var p1 = new Point(1, 2);
var p2 = new Point(1, 2);
System.out.println("p1.equals(p2) = " + p1.equals(p2));

Map<ProductKey, Integer> stock = new HashMap<>();
stock.put(new ProductKey("SKU-1", "AZ"), 12);
System.out.println("SKU-1/AZ qalığı = " + stock.get(new ProductKey("SKU-1", "AZ")));

## Nested, Local Record və Pattern Matching

In [ ]:
record OrderEvent(String orderId, String status) {}

class ReportService {
    record Row(String label, long count) {}
}

var row = new ReportService.Row("Paid Orders", 17);
System.out.println("Nested record: " + row);

record Tmp(int id, String name) {}
var t = new Tmp(1, "Demo");
System.out.println("Local record: " + t);

Object obj = new OrderEvent("ORD-1", "PAID");
if (obj instanceof OrderEvent(String orderId, String status)) {
    System.out.println("Pattern matching: " + orderId + " -> " + status);
}

## sealed + record

In [ ]:
sealed interface PaymentResult permits Success, Failure {}
record Success(String txId) implements PaymentResult {}
record Failure(String reason) implements PaymentResult {}

void printResult(PaymentResult r) {
    if (r instanceof Success s) System.out.println("SUCCESS txId=" + s.txId());
    else if (r instanceof Failure f) System.out.println("FAILURE reason=" + f.reason());
}

printResult(new Success("TX-1001"));
printResult(new Failure("Insufficient balance"));

## Real Nümunə: CheckoutResponse

In [ ]:
import java.math.BigDecimal;
import java.time.Instant;

record CheckoutResponse(String orderId, BigDecimal total, String currency, String status, Instant createdAt) {
    public CheckoutResponse {
        if (orderId == null || orderId.isBlank()) throw new IllegalArgumentException("orderId boş ola bilməz");
        if (total == null || total.signum() < 0) throw new IllegalArgumentException("məbləğ mənfi ola bilməz");
        if (currency == null || currency.isBlank()) throw new IllegalArgumentException("currency boş ola bilməz");
        if (status == null || status.isBlank()) throw new IllegalArgumentException("status boş ola bilməz");
    }
}

System.out.println(new CheckoutResponse("ORD-2026-0001", new BigDecimal("149.90"), "AZN", "PAID", Instant.now()));
try {
    new CheckoutResponse("", new BigDecimal("-1"), "AZN", "PAID", Instant.now());
} catch (Exception e) {
    System.out.println("Validation xətası: " + e.getMessage());
}